# Lexical-gap project: dense retrieval on GPU
1. **Runtime > Change runtime type > T4 GPU**, then Save.
2. **Runtime > Run all**. It takes about 30-45 min.
3. The last cell downloads `results_colab.zip`; upload that file back into the Claude chat.

In [ ]:
!nvidia-smi -L
!pip -q install sentence-transformers PyStemmer
import os; os.makedirs('/content/proj/src', exist_ok=True); os.makedirs('/content/proj/data', exist_ok=True)

In [ ]:
%%writefile /content/proj/src/common.py
"""Shared utilities: data loading, tokenisation/indexing, evaluation, run I/O.

All randomness is driven by SEED. Paths are relative to the project root.
"""
import json
import os
import re
import pickle
from collections import defaultdict

import numpy as np
import scipy.sparse as sp
import Stemmer
from sklearn.feature_extraction.text import CountVectorizer

SEED = 42
ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DATA = os.path.join(ROOT, "data")
CACHE = os.path.join(ROOT, "cache")
RES = os.path.join(ROOT, "results")
FIG = os.path.join(ROOT, "figures")
for _d in (CACHE, RES, FIG):
    os.makedirs(_d, exist_ok=True)

DATASETS = ["nfcorpus", "scifact", "fiqa", "scidocs", "arguana",
            "trec-covid", "quora", "webis-touche2020"]
CORE = ["nfcorpus", "scifact", "fiqa"]  # the three datasets of the original proposal
DOMAIN = {"nfcorpus": "biomedical", "scifact": "scientific", "fiqa": "financial",
          "scidocs": "scientific", "arguana": "argument", "trec-covid": "biomedical",
          "quora": "general (dup. questions)", "webis-touche2020": "argument"}
PRETTY = {"nfcorpus": "NFCorpus", "scifact": "SciFact", "fiqa": "FiQA-2018",
          "scidocs": "SciDocs", "arguana": "ArguAna", "trec-covid": "TREC-COVID",
          "quora": "Quora", "webis-touche2020": "Touche-2020"}

# Lucene's default English stopword set (what Pyserini/Anserini use for BEIR BM25).
LUCENE_STOP = set("""a an and are as at be but by for if in into is it no not of on or
such that the their then there these they this to was will with""".split())

TOKEN_RE = re.compile(r"\w+", re.UNICODE)
_stemmer = Stemmer.Stemmer("porter")


# ----------------------------------------------------------------------------- data
def load_dataset(name):
    d = os.path.join(DATA, name)
    doc_ids, texts = [], []
    with open(os.path.join(d, "corpus.jsonl")) as f:
        for line in f:
            o = json.loads(line)
            doc_ids.append(str(o["_id"]))
            texts.append((o.get("title") or "") + " " + (o.get("text") or ""))
    queries = {}
    with open(os.path.join(d, "queries.jsonl")) as f:
        for line in f:
            o = json.loads(line)
            queries[str(o["_id"])] = o["text"]
    qrels = defaultdict(dict)
    with open(os.path.join(d, "qrels", "test.tsv")) as f:
        next(f)
        for line in f:
            q, doc, s = line.rstrip("\n").split("\t")
            qrels[q][doc] = int(s)
    # keep only queries that have at least one positive judgement (BEIR/trec_eval practice)
    qrels = {q: r for q, r in qrels.items() if any(v > 0 for v in r.values()) and q in queries}
    queries = {q: queries[q] for q in qrels}
    return doc_ids, texts, queries, qrels


def tokenize(text):
    return TOKEN_RE.findall(text.lower())


# ----------------------------------------------------------------------------- index
class Index:
    """Raw count matrix built once; stem/stop variants derived by column mapping."""

    def __init__(self, name):
        self.name = name
        path = os.path.join(CACHE, f"{name}_index.pkl")
        if os.path.exists(path):
            with open(path, "rb") as f:
                self.__dict__.update(pickle.load(f))
            return
        doc_ids, texts, queries, qrels = load_dataset(name)
        cv = CountVectorizer(lowercase=True, token_pattern=r"(?u)\w+", dtype=np.int32)
        C = cv.fit_transform(texts).tocsr()
        raw_vocab = cv.get_feature_names_out().tolist()
        self.doc_ids, self.queries, self.qrels = doc_ids, queries, qrels
        self.raw_vocab = raw_vocab
        self.C_raw = C
        with open(path, "wb") as f:
            pickle.dump(self.__dict__, f, protocol=4)

    def variant(self, stem=True, stop=True):
        """Return (doc-term count matrix, vocab dict term->col, analyser fn)."""
        key = (stem, stop)
        if not hasattr(self, "_variants"):
            self._variants = {}
        if key in self._variants:
            return self._variants[key]
        raw = self.raw_vocab
        terms = _stemmer.stemWords(raw) if stem else raw
        keep = np.array([t not in LUCENE_STOP for t in raw]) if stop else np.ones(len(raw), bool)
        vocab = {}
        rows, cols = [], []
        for i, (t, k) in enumerate(zip(terms, keep)):
            if not k:
                continue
            j = vocab.setdefault(t, len(vocab))
            rows.append(i)
            cols.append(j)
        M = sp.csr_matrix((np.ones(len(rows), np.int32), (rows, cols)), shape=(len(raw), len(vocab)))
        C = (self.C_raw @ M).tocsr()
        C.sort_indices()

        def analyse(text, _stem=stem, _stop=stop):
            toks = tokenize(text)
            if _stop:
                toks = [t for t in toks if t not in LUCENE_STOP]
            if _stem:
                toks = _stemmer.stemWords(toks)
            return toks

        self._variants[key] = (C, vocab, analyse)
        return self._variants[key]


def query_matrix(qtexts, vocab, analyse, binary=False):
    rows, cols, vals = [], [], []
    for i, t in enumerate(qtexts):
        counts = defaultdict(int)
        for tok in analyse(t):
            j = vocab.get(tok)
            if j is not None:
                counts[j] += 1
        for j, c in counts.items():
            rows.append(i)
            cols.append(j)
            vals.append(1.0 if binary else float(c))
    return sp.csr_matrix((vals, (rows, cols)), shape=(len(qtexts), len(vocab)), dtype=np.float32)


# ----------------------------------------------------------------------------- weighting
def bm25_matrix(C, k1=0.9, b=0.4):
    """Lucene-style BM25 doc weights: idf * tf*(k1+1)/(tf + k1*(1-b+b*dl/avgdl))."""
    C = C.tocsr().astype(np.float32)
    N = C.shape[0]
    df = np.bincount(C.indices, minlength=C.shape[1])
    idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.asarray(C.sum(1)).ravel()
    avgdl = dl.mean()
    W = C.copy()
    rep_dl = np.repeat(dl, np.diff(C.indptr))
    tf = W.data
    W.data = tf * (k1 + 1) / (tf + k1 * (1 - b + b * rep_dl / avgdl)) * idf[W.indices]
    return W, idf


def tfidf_matrix(C):
    """SMART ltc: (1+log tf)*log(N/df), cosine-normalised. Returns (W, idf)."""
    C = C.tocsr().astype(np.float32)
    N = C.shape[0]
    df = np.bincount(C.indices, minlength=C.shape[1])
    idf = np.log(N / np.maximum(df, 1)).astype(np.float32)
    W = C.copy()
    W.data = (1 + np.log(W.data)) * idf[W.indices]
    norms = np.sqrt(np.asarray(W.multiply(W).sum(1)).ravel())
    norms[norms == 0] = 1
    W = sp.diags(1 / norms) @ W
    return W.tocsr(), idf


def search(W, Q, k=1000, batch=128, tie_noise=None, min_score=None):
    """Score all docs: S = Q @ W^T. Returns list of (doc_idx array, score array).

    tie_noise: if given, add uniform noise of that magnitude (seeded) to break ties
    randomly (used for Boolean models whose scores are integers).
    min_score: per-query array; docs scoring below it are not retrieved (Boolean AND).
    """
    rng = np.random.default_rng(SEED)
    WT = W.T.tocsr()
    out = []
    for s in range(0, Q.shape[0], batch):
        Qb = Q[s:s + batch]
        S = np.asarray((Qb @ WT).todense(), dtype=np.float32)
        for i in range(S.shape[0]):
            raw = S[i]
            if min_score is not None:
                valid = np.where(raw >= min_score[s + i] - 1e-6)[0]
            else:
                valid = np.where(raw > 0)[0]
            row = raw
            if tie_noise and len(valid):
                row = raw.copy()
                row[valid] += rng.uniform(0, tie_noise, len(valid)).astype(np.float32)
            kk = min(k, len(valid))
            if kk == 0:
                out.append((np.array([], int), np.array([], np.float32)))
                continue
            sub = row[valid]
            top = np.argpartition(-sub, kk - 1)[:kk]
            top = top[np.argsort(-sub[top], kind="stable")]
            out.append((valid[top], sub[top]))
    return out


def to_run(results, qids, doc_ids, drop_identical=True):
    """Convert search output to {qid: [(docid, score), ...]} (BEIR ignore_identical_ids)."""
    run = {}
    for q, (idx, sc) in zip(qids, results):
        lst = [(doc_ids[i], float(s)) for i, s in zip(idx, sc)]
        if drop_identical:
            lst = [x for x in lst if x[0] != q]
        run[q] = lst
    return run


# ----------------------------------------------------------------------------- evaluation
def evaluate_query(ranked, rels):
    """Per-query metrics. ranked: list of docids; rels: {docid: grade}. Linear gains (trec_eval)."""
    pos = {d: g for d, g in rels.items() if g > 0}
    nrel = len(pos)
    gains = [pos.get(d, 0) for d in ranked]
    # nDCG@10
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains[:10]))
    ideal = sorted(pos.values(), reverse=True)[:10]
    idcg = sum(g / np.log2(i + 2) for i, g in enumerate(ideal))
    ndcg = dcg / idcg if idcg > 0 else 0.0
    # MRR@10
    mrr = 0.0
    for i, g in enumerate(gains[:10]):
        if g > 0:
            mrr = 1 / (i + 1)
            break
    # Recall@100, P@10
    hits100 = sum(1 for g in gains[:100] if g > 0)
    rec = hits100 / nrel
    p10 = sum(1 for g in gains[:10] if g > 0) / 10
    # MAP@100 (trec_eval map_cut_100)
    h, ap = 0, 0.0
    for i, g in enumerate(gains[:100]):
        if g > 0:
            h += 1
            ap += h / (i + 1)
    ap /= nrel
    rec1000 = sum(1 for g in gains[:1000] if g > 0) / nrel
    return {"ndcg@10": ndcg, "map@100": ap, "recall@100": rec, "mrr@10": mrr,
            "p@10": p10, "recall@1000": rec1000}


def evaluate_run(run, qrels):
    per = {}
    for q, rels in qrels.items():
        ranked = [d for d, _ in run.get(q, [])]
        per[q] = evaluate_query(ranked, rels)
    return per


def save_run(run, dataset, system, k=1000):
    d = os.path.join(CACHE, "runs")
    os.makedirs(d, exist_ok=True)
    with open(os.path.join(d, f"{dataset}__{system}.pkl"), "wb") as f:
        pickle.dump({q: v[:k] for q, v in run.items()}, f, protocol=4)


def load_run(dataset, system):
    with open(os.path.join(CACHE, "runs", f"{dataset}__{system}.pkl"), "rb") as f:
        return pickle.load(f)


def per_query_rows(per, dataset, system):
    return [dict(dataset=dataset, system=system, qid=q, **m) for q, m in per.items()]


In [ ]:
%%writefile /content/proj/src/run_dense.py
"""E6: dense retrieval (exact cosine search) and BM25+dense hybrids.

Usage: python src/run_dense.py <model_key> [dataset ...]
model_key in {minilm, bge}. Requires run_lexical.py to have produced the bm25 run.
"""
import os
import sys
import time

import numpy as np
import pandas as pd
import torch

from common import (CACHE, DATASETS, RES, load_dataset, evaluate_run, per_query_rows,
                    save_run, load_run)

MODELS = {"minilm": ("sentence-transformers/all-MiniLM-L6-v2", ""),
          "bge": ("BAAI/bge-small-en-v1.5", "Represent this sentence for searching relevant passages: ")}
torch.set_num_threads(int(os.environ.get("THREADS", "4")))


def encode(model, texts, path, bs=int(os.environ.get("BATCH", "64"))):
    if os.path.exists(path):
        return np.load(path).astype(np.float32)
    t = time.time()
    E = model.encode(texts, batch_size=bs, normalize_embeddings=True, show_progress_bar=False,
                     convert_to_numpy=True)
    np.save(path, E.astype(np.float16))
    print(f"    encoded {len(texts)} in {time.time()-t:.0f}s", flush=True)
    return E.astype(np.float32)


def dense_search(Dm, Qm, k=1000, batch=256):
    out = []
    for s in range(0, len(Qm), batch):
        S = Qm[s:s + batch] @ Dm.T
        for row in S:
            kk = min(k, len(row))
            top = np.argpartition(-row, kk - 1)[:kk]
            top = top[np.argsort(-row[top])]
            out.append((top, row[top]))
    return out


def rrf(runs, k=60, depth=1000):
    fused = {}
    for q in runs[0]:
        sc = {}
        for r in runs:
            for rank, (d, _) in enumerate(r.get(q, [])[:depth]):
                sc[d] = sc.get(d, 0) + 1 / (k + rank + 1)
        fused[q] = sorted(sc.items(), key=lambda x: -x[1])[:depth]
    return fused


def convex(r1, r2, alpha=0.5, depth=1000):
    """alpha*minmax(r1) + (1-alpha)*minmax(r2); unseen docs get 0 from that system."""
    fused = {}
    for q in r1:
        sc = {}
        for r, w in ((r1, alpha), (r2, 1 - alpha)):
            lst = r.get(q, [])[:depth]
            if not lst:
                continue
            v = np.array([s for _, s in lst])
            lo, hi = v.min(), v.max()
            for d, s in lst:
                sc[d] = sc.get(d, 0) + w * ((s - lo) / (hi - lo) if hi > lo else 1.0)
        fused[q] = sorted(sc.items(), key=lambda x: -x[1])[:depth]
    return fused


def main(mkey, name, model):
    print(f"=== {mkey} {name}", flush=True)
    doc_ids, texts, queries, qrels = load_dataset(name)
    qids = list(qrels)
    prefix = MODELS[mkey][1] if name != "quora" else ""
    D = encode(model, texts, os.path.join(CACHE, f"emb_{mkey}_{name}_docs.npy"))
    Qm = encode(model, [prefix + queries[q] for q in qids], os.path.join(CACHE, f"emb_{mkey}_{name}_q.npy"))
    res = dense_search(D, Qm)
    run = {q: [(doc_ids[i], float(s)) for i, s in zip(idx, sc) if doc_ids[i] != q] for q, (idx, sc) in zip(qids, res)}
    save_run(run, name, mkey)
    rows = []
    per = evaluate_run(run, qrels)
    rows += per_query_rows(per, name, mkey)
    bm25 = load_run(name, "bm25")
    for sysname, r in ((f"hybrid_rrf_{mkey}", rrf([bm25, run])),
                       (f"hybrid_cc_{mkey}", convex(bm25, run))):
        save_run(r, name, sysname)
        rows += per_query_rows(evaluate_run(r, qrels), name, sysname)
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(RES, f"perquery_{mkey}_{name}.csv"), index=False)
    print(df.groupby("system")["ndcg@10"].mean().round(4).to_string(), flush=True)


if __name__ == "__main__":
    from sentence_transformers import SentenceTransformer
    mkey = sys.argv[1]
    model = SentenceTransformer(MODELS[mkey][0], device=os.environ.get("DEVICE", "cpu"))
    model.max_seq_length = int(os.environ.get("MAXLEN", "256"))
    for d in (sys.argv[2:] or DATASETS):
        main(mkey, d, model)


In [ ]:
%%writefile /content/proj/colab_main.py
"""Colab GPU runner: BM25 (needed for hybrids) + MiniLM + bge-small dense + hybrids on all 8 datasets.
Produces results_colab.zip containing per-query CSVs to upload back for analysis."""
import os, sys, zipfile, subprocess, time, json, platform
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "src"))
os.environ.setdefault("DEVICE", "cuda")
os.environ.setdefault("BATCH", "256")
os.environ.setdefault("MAXLEN", "256")
import numpy as np, pandas as pd, torch
from common import DATASETS, DATA, RES, Index, bm25_matrix, query_matrix, search, to_run, evaluate_run, save_run, per_query_rows
import run_dense
from sentence_transformers import SentenceTransformer

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{}.zip"
for d in DATASETS:
    if not os.path.exists(os.path.join(DATA, d, "corpus.jsonl")):
        print("downloading", d, flush=True)
        subprocess.run(f"cd {DATA} && wget -q {URL.format(d)} && unzip -q -o {d}.zip && rm {d}.zip", shell=True, check=True)

check = []
for d in DATASETS:  # BM25 run, identical code/params to the main pipeline
    t = time.time()
    idx = Index(d)
    C, vocab, an = idx.variant(True, True)
    qids = list(idx.qrels)
    Q = query_matrix([idx.queries[q] for q in qids], vocab, an)
    W, _ = bm25_matrix(C)
    run = to_run(search(W, Q), qids, idx.doc_ids)
    save_run(run, d, "bm25")
    m = np.mean([v["ndcg@10"] for v in evaluate_run(run, idx.qrels).values()])
    check.append(dict(dataset=d, bm25_ndcg10=m))
    print(f"bm25 {d}: {m:.4f} ({time.time()-t:.0f}s)", flush=True)
    del idx, C, W, Q
pd.DataFrame(check).to_csv(os.path.join(RES, "colab_bm25_check.csv"), index=False)

for mkey in ["minilm", "bge"]:
    model = SentenceTransformer(run_dense.MODELS[mkey][0], device="cuda")
    model.max_seq_length = int(os.environ["MAXLEN"])
    for d in DATASETS:
        t = time.time()
        run_dense.main(mkey, d, model)
        print(f"  {mkey} {d} done in {time.time()-t:.0f}s", flush=True)

with open(os.path.join(RES, "colab_env.json"), "w") as f:
    json.dump(dict(gpu=torch.cuda.get_device_name(0), torch=torch.__version__, python=platform.python_version(),
                   maxlen=os.environ["MAXLEN"], batch=os.environ["BATCH"]), f, indent=1)
out = "/content/results_colab.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in os.listdir(RES):
        if fn.startswith(("perquery_minilm", "perquery_bge", "colab_")):
            z.write(os.path.join(RES, fn), fn)
print("WROTE", out)


In [ ]:
!cd /content/proj && python colab_main.py 2>&1 | grep -v -i 'warn\|loading weights'

In [ ]:
from google.colab import files
files.download('/content/results_colab.zip')